1. imports e carregar candidatos

In [1]:
import csv
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool

load_dotenv()

with open("../data/candidatos/candidatos.csv", encoding="utf-8-sig") as f:
    candidatos = list(csv.DictReader(f)) #perguntar

# dicionário: nome comum -> id_candidato (o que está salvo no metadado do Chroma)
nome_para_id = {c["nome_urna"]: c["id_candidato"] for c in candidatos}

print(nome_para_id)

{'Lula': 'lula', 'Renan Santos': 'renan_santos', 'Hertz Dias': 'hertz_dias', 'Edmilson Costa': 'edmilson_costa', 'Flavio Bolsonaro': 'flavio_bolsonaro', 'Clariana Barao': 'clariana_barao', 'Pablo Marçal': 'pablo_marcal', 'Rui Costa Pimenta': 'rui_costa_pimenta', 'Zema': 'zema', 'Veterinário Wilson Grassi': 'veterinario_wilson_grassi', 'Ronaldo Caiado': 'ronaldo_caiado', 'Escritor Augusto Cury': 'escritor_augusto_cury', 'Samara': 'samara'}


2. carregar o banco vetorial já existente

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

vectorstore = Chroma(
    persist_directory="../data/vectorstores/chroma_candidatos",
    embedding_function=embeddings,
)

print("Banco vetorial carregado.")

Banco vetorial carregado.


3. a tool

In [3]:
@tool
def buscar_propostas_candidato(candidato: str, pergunta: str) -> str:
    """Busca trechos das propostas de governo de um candidato à presidência do Brasil em 2026.

    Use esta ferramenta sempre que o usuário perguntar sobre propostas, planos de governo
    ou posições de um candidato específico em algum tema (educação, saúde, economia, etc).

    Args:
        candidato: o nome comum do candidato, exatamente como é conhecido (ex: "Lula", "Renan Santos").
        pergunta: a pergunta ou tema a buscar dentro do documento de propostas desse candidato.
    """
    id_candidato = nome_para_id.get(candidato)

    if id_candidato is None:
        candidatos_disponiveis = ", ".join(nome_para_id.keys())
        return f"Candidato '{candidato}' não encontrado. Candidatos disponíveis: {candidatos_disponiveis}"

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 4, "filter": {"candidato": id_candidato}}
    )
    resultados = retriever.invoke(pergunta)

    if not resultados:
        return f"Nenhum trecho encontrado sobre '{pergunta}' nas propostas de {candidato}."

    trechos_formatados = []
    for r in resultados:
        pagina = r.metadata["page"] + 1
        nome = r.metadata["nome_urna"]
        trechos_formatados.append(f"[{nome} - Página {pagina}] {r.page_content}")

    return "\n\n".join(trechos_formatados)

4. testar a tool isoladamente (sem LLM ainda, só a função)

In [4]:
resultado = buscar_propostas_candidato.invoke({
    "candidato": "Renan Santos",
    "pergunta": "propostas para educação"
})

print(resultado)

[Renan Santos - Página 31] portamental do estudante, com implicações punitivas, de modo a restaurar o sentido disciplinar dentro das 
escolas. Precisaremos abrir diálogo amplo com a comunidade pedagógica para que esta admita o problema e 
busque contribuir em suas soluções efetivas.
Outra proposta é mitigar a progressão continuada nas escolas em que ela ainda vigora integralmente. O 
instituto da progressão continuada, criado para dirimir o problema da evasão escolar, tem criado um outro 
problema correlato: alunos que passam de ano sem domínio efetivo do conteúdo escolar ministrado. Deve-
mos criar outros instrumentos para a retenção do aluno em sala para além da progressão.
Por fim, temos no horizonte uma ambiciosa reforma do ensino superior, dentro da qual devemos tam-
bém enfrentar uma série de questões centrais, que tensionarão esta relação ainda mais: devemos realocar os 
recursos atualmente investidos em cursos bacharelescos para a área de STEM; substituir o sistema de cotas

[R

5. a tool de busca web

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

busca_duckduckgo = DuckDuckGoSearchRun()

@tool
def buscar_informacoes_web(candidato: str, tema: str) -> str:
    """Busca na internet informações atuais sobre um candidato à presidência do Brasil em 2026.

    Use esta ferramenta apenas para perguntas que NÃO sejam sobre propostas de governo
    (isso já é coberto pela ferramenta buscar_propostas_candidato). Prefira esta ferramenta
    para: notícias recentes, biografia, trajetória política, ou quando a busca nas propostas
    não encontrou nada sobre o tema perguntado.

    Args:
        candidato: nome comum do candidato, exatamente como conhecido (ex: "Lula", "Renan Santos").
        tema: o que buscar sobre esse candidato (ex: "biografia", "últimas notícias", "trajetória política").
    """
    if candidato not in nome_para_id:
        candidatos_disponiveis = ", ".join(nome_para_id.keys())
        return f"'{candidato}' não é um dos candidatos à presidência cobertos por este projeto. Candidatos disponíveis: {candidatos_disponiveis}"

    query = f"{candidato} candidato presidente Brasil eleições 2026 {tema}" 
    return busca_duckduckgo.invoke(query)

6. teste isolado

In [7]:
resultado = buscar_informacoes_web.invoke({
    "candidato": "Renan Santos",
    "tema": "trajetória política"
})

print(resultado)

3 days ago - ↑ "Com discurso firme contra o PT e o crime, Renan Santos lança pré-candidatura à Presidência pelo "Missão"". BRADO JORNAL (in Brazilian Portuguese). Retrieved July 21, 2026. ↑ "Candidato do MBL a presidente fica em segundo entre mais jovens em pesquisa | Política Livre". 1 week ago - A campanha presidencial de Renan ... ativista político, coordenador do Movimento Brasil Livre (MBL) e líder partidário na disputa pelo Palácio do Planalto. O Partido Missão antecipou sua convenção nacional em julho de 2026 para oficializar o nome de Renan como candidato à Presidência da República, tendo como candidato a vice-presidente o oficial ... 5 days ago - Conheça Renan Santos, número 14, candidato a Presidente nas eleições 2026. Veja informações sobre o candidato, propostas e trajetória política. Renan é candidato à presidência pelo Partido Missão. Conheça mais e descubra porque o futuro é glorioso! 1 month ago - A trajetória que o levou até esse ponto mistura política estudantil, recu